# Hybrid Transformer vs. Standard Transformer Comparison

This notebook compares the standard Probabilistic Transformer against the Hybrid Transformer which integrates an Ornstein-Uhlenbeck (OU) process to model residuals

## Methodology

1.  Standard transformer: predicting prices solely based on input features
2.  Hybrid Transformer:
    - Train Transformer for trend prediction
    - Model residuals (Actual - Predicted) using an OU process
    - Final forecast = transformer trend + OU process mean reversion

We run both models 10 times to reduce statistical variability and report average metrics (MAE, RMSE, MAPE, R2, Pinball Loss)

In [6]:
import os
import sys
import json
import time
import gc
import warnings
from pathlib import Path
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

current_dir = Path.cwd()
project_root = None
if (current_dir / 'config.py').exists():
    project_root = str(current_dir)
elif (current_dir.parent / 'config.py').exists():
    project_root = str(current_dir.parent)

if project_root and project_root not in sys.path:
    sys.path.insert(0, project_root)

from config import ExperimentConfig, DataConfig, ModelConfig, TrainingConfig
from data import DataPipeline, DataLoader
from models import ProbabilisticTransformer, HybridProbabilisticTransformer
from core.trainer import Trainer
from transformations import StandardScalingTransformation

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

try:
    gpus = tf.config.experimental.list_physical_devices("GPU")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPUs detected: {len(gpus)}")
except Exception as e:
    print(f"GPU config failed: {e}")

GPUs detected: 1


In [7]:
# Configuration
RESULTS_DIR = Path(project_root) / "results" / "hybrid_comparison"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = RESULTS_DIR / "results.json"

N_RUNS = 10
INPUT_WINDOW = 168
OUTPUT_HORIZON = 24

BASE_MODEL_CONFIG = {
    "d_model": 128,
    "num_heads": 4,
    "num_layers": 2,
    "ff_dim": 256,
    "dropout": 0.1,
}

TRAIN_SETTINGS = {
    "batch_size": 64,
    "epochs": 30,
    "learning_rate": 1e-3,
    "patience": 5,
}

In [8]:
def create_windows(data, input_window, output_horizon):
    X, y = [], []
    for i in range(len(data) - input_window - output_horizon + 1):
        X.append(data[i:(i + input_window)])
        y.append(data[(i + input_window):(i + input_window + output_horizon), 0])
    return np.array(X), np.array(y)

def calculate_metrics(y_true, y_pred):
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()
    
    mae = mean_absolute_error(y_true_flat, y_pred_flat)
    rmse = np.sqrt(mean_squared_error(y_true_flat, y_pred_flat))
    r2 = r2_score(y_true_flat, y_pred_flat)
    
    mask = y_true_flat != 0
    mape = np.mean(np.abs((y_true_flat[mask] - y_pred_flat[mask]) / y_true_flat[mask])) * 100
    
    return {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "MAPE": float(mape),
        "R2": float(r2)
    }

def pinball_loss(y_true, y_pred_q, q):
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred_q.flatten()
    err = y_true_flat - y_pred_flat
    return np.mean(np.maximum(q * err, (q - 1.0) * err))

def load_cache() -> Dict[str, Any]:
    if CACHE_FILE.exists():
        try:
            with open(CACHE_FILE, "r") as f:
                return json.load(f)
        except json.JSONDecodeError:
            return {}
    return {}

def save_cache(cache: Dict[str, Any]) -> None:
    with open(CACHE_FILE, "w") as f:
        json.dump(cache, f, indent=2)

In [9]:
# Load and preprocess data
print("Loading Data...")
data_config = DataConfig(dataset_name="BE_ENTSOE")
pipeline = DataPipeline(data_config)
df_train, df_val, df_test = pipeline.get_data_splits()

# Fill NaN values in exogenous features
df_train = df_train.ffill().bfill()
df_val = df_val.ffill().bfill()
df_test = df_test.ffill().bfill()

print(f"NaN remaining - Train: {df_train.isnull().sum().sum()}, Val: {df_val.isnull().sum().sum()}, Test: {df_test.isnull().sum().sum()}")

# Create windows
X_train, y_train = create_windows(df_train.values, INPUT_WINDOW, OUTPUT_HORIZON)
X_val, y_val = create_windows(df_val.values, INPUT_WINDOW, OUTPUT_HORIZON)
X_test_raw, y_test_raw = create_windows(df_test.values, INPUT_WINDOW, OUTPUT_HORIZON)

# scaled data for training
scaler = StandardScalingTransformation()
scaler.fit(X_train, y_train)
X_train_s, y_train_s = scaler.transform(X_train, y_train)
X_val_s, y_val_s = scaler.transform(X_val, y_val)
X_test_s, y_test_s = scaler.transform(X_test_raw, y_test_raw)


Loading Data...
NaN remaining - Train: 0, Val: 0, Test: 0


In [10]:
cache = load_cache()
models_to_run = ["Standard Transformer", "Hybrid (Transformer+OU)"]

for model_name in models_to_run:
    if model_name in cache:
        print(f"Results for {model_name} found in cache.")
        continue
        
    print(f"\nRunning Experiments for: {model_name}")
    run_metrics = []
    
    for r in range(N_RUNS):
        print(f"  Run {r+1}/{N_RUNS}...")
        tf.keras.backend.clear_session()
        gc.collect()
        
        # Config setup
        model_conf = ModelConfig(**BASE_MODEL_CONFIG)
        train_conf = TrainingConfig(**TRAIN_SETTINGS)
        dc = DataConfig("BE_ENTSOE", input_window=INPUT_WINDOW, output_horizon=OUTPUT_HORIZON)
        exp_conf = ExperimentConfig(
            name=f"{model_name}_run{r}",
            data_config=dc,
            model_config=model_conf,
            training_config=train_conf,
            head_type="johnson_su"
        )
        
        if "Hybrid" in model_name:
            model = HybridProbabilisticTransformer(exp_conf)
        else:
            model = ProbabilisticTransformer(exp_conf)
            
        # Train
        trainer = Trainer(exp_conf)
        try:
            trainer.train(model, X_train_s, y_train_s, X_val_s, y_val_s)
        except Exception as e:
            print(f"    Training Error: {e}")
            continue
            
        if "Hybrid" in model_name:
            model.fit_ou(X_train_s, y_train_s)
            # Prediction (mean)
            y_pred_scaled = model.predict_hybrid(X_test_s, last_residuals=None)
            # Quantiles (0.1, 0.5, 0.9)
            q_scaled_dict = model.quantiles_hybrid(X_test_s, [0.1, 0.5, 0.9], n_samples=200)
            
        else:
            # Standard predict
            y_params = model.keras_model.predict(X_test_s, verbose=0)
            flat_params = y_params.reshape(-1, y_params.shape[-1])
            
            # Mean
            y_pred_scaled_flat = model.head.mean(flat_params)
            y_pred_scaled = y_pred_scaled_flat.reshape(X_test_s.shape[0], -1)
            
            # Quantiles
            q_dict = model.head.quantiles(flat_params, [0.1, 0.5, 0.9])
            q_scaled_dict = {
                0.1: q_dict[0.1].reshape(y_pred_scaled.shape),
                0.5: q_dict[0.5].reshape(y_pred_scaled.shape),
                0.9: q_dict[0.9].reshape(y_pred_scaled.shape)
            }

        # Inverse transform mean
        _, y_pred = scaler.inverse_transform(None, y_pred_scaled)
        
        # Inverse transform quantiles
        q_raw_dict = {}
        for q, val_scaled in q_scaled_dict.items():
             _, val_raw = scaler.inverse_transform(None, val_scaled)
             q_raw_dict[q] = val_raw
        
        # Calculate metrics
        m = calculate_metrics(y_test_raw, y_pred)
        
        # Add pinball losses
        m["Pinball_10"] = pinball_loss(y_test_raw, q_raw_dict[0.1], 0.1)
        m["Pinball_50"] = pinball_loss(y_test_raw, q_raw_dict[0.5], 0.5)
        m["Pinball_90"] = pinball_loss(y_test_raw, q_raw_dict[0.9], 0.9)
        m["Avg_Pinball"] = (m["Pinball_10"] + m["Pinball_50"] + m["Pinball_90"]) / 3.0
        
        run_metrics.append(m)
        
    if not run_metrics: continue
    
    # Average metrics
    avg_metrics = {}
    for k in run_metrics[0].keys():
        avg_metrics[k] = np.mean([rm[k] for rm in run_metrics])
        
    cache[model_name] = avg_metrics
    save_cache(cache)
    print(f"  Average MAE: {avg_metrics['MAE']:.4f}")

Results for Standard Transformer found in cache.
Results for Hybrid (Transformer+OU) found in cache.


In [11]:
results = []
for k, v in cache.items():
    results.append({"Model": k, **v})
    
df_res = pd.DataFrame(results)
if not df_res.empty:
    display(df_res.sort_values("MAE"))
    df_res.to_csv(RESULTS_DIR / "comparison_summary.csv", index=False)

,Model,MAE,RMSE,MAPE,R2,Pinball_10,Pinball_50,Pinball_90,Avg_Pinball
1,Hybrid (Transformer+OU),24.575903,33.169699,840.671938,0.167225,5.747766,12.109231,6.503489,8.120162
0,Standard Transformer,25.924237,34.583847,834.891360,0.093906,6.772049,12.669376,7.406705,8.949377
